# The Vol-Gordian Project — Colab Pipeline
## Volatility-Managed Long Exposure

Successor to **The Gordian Project**, whose directional-alpha thesis was disproven out-of-sample (OOS win rate ~50% across 3 experiments; PBO/CSCV = 0.65 FAIL). Direction is near-unpredictable at short horizons; **volatility is forecastable**. So instead of betting on *which way* the market moves, we size a LONG position inversely to forecast volatility.

### The hypothesis (Moreira & Muir, 2017)
Volatility is persistent and forecastable, while high-volatility periods do **not** pay proportionally higher returns. Scaling exposure down when vol is high therefore improves risk-adjusted return and cuts drawdown versus static buy-and-hold.

### ⚠️ The success metric has changed
**Judge this on Sharpe ratio and max drawdown vs Buy-and-Hold — NOT raw ROI.** The strategy is de-risked on average, so its raw return may *trail* buy-and-hold in a bull market; that is expected and fine. This is a **beta + risk-management product, not directional alpha.**

### Pipeline
1. Setup & clone
2. Data engineering (Information-Driven Dollar Bars + `Close`)
3. Volatility-managed backtest vs Buy-and-Hold
4. Read results & next steps

**Runtime:** ~20–30 min (mostly data fetch). CPU only — no GPU, no ML.

---
## STEP 1: Setup & Clone

Clones the repo and installs dependencies (pure pandas/numpy stack, no ML/RL libraries for the baseline).

In [ ]:
# ==========================================
# STEP 1: SETUP & CLONE
# ==========================================
%cd /content
!rm -rf gordian-vol-targeting
!git clone https://github.com/Aymo-WH/gordian-vol-targeting.git

%cd gordian-vol-targeting
!pip install -q -r requirements.txt
print("\n\u2713 Repo cloned and dependencies installed.")

---
## STEP 2: Data Engineering

Builds **Information-Driven Dollar Bars** (sampled on dollar volume, not clock time — neutralizes heteroscedasticity) and computes `Close` per asset.

> The volatility baseline only needs `Close`. The extra microstructural/price features and PCA are **inherited from Gordian** and will be useful later *if* we add an ML volatility forecast. (The data engine will be trimmed down the line.)

**Universe** (`config/config_phase1.json`): SPY, QQQ, XLK, XLF, XLE, BTC-USD, ETH-USD, TQQQ.
(The run written up in the README predates SPY and QQQ and covered the other six.)
VXX is **excluded** — a vol-managed *long* needs positive expected drift, and VXX structurally decays.

**Output:** `data/train/{ticker}_data.csv`, `data/test/{ticker}_data.csv` (80/20 split, 60-bar embargo).

In [ ]:
# ==========================================
# STEP 2: DATA ENGINEERING (Dollar Bars + Close)
# ==========================================
print("--- BUILDING DATA ---")
!python src/data_factory.py --config config/config_phase1.json

print("\n\u2713 Data built.")
!ls -lh data/test/

---
## STEP 3: Volatility-Managed Backtest

**The rule** (one risk parameter → near-zero overfitting surface):
```
w_t = clip( target_vol / forecast_vol_t , 0, 1 )      # long-only, no leverage
  forecast_vol_t = EWMA realised vol using returns up to t-1   (no look-ahead)
  target_vol     = median EWMA vol on the TRAIN split          (no test leakage)
  strategy_return_t = w_t * r_t  -  |w_t - w_{t-1}| * fee
```
Hold **less** when volatility is forecast high, **more** when calm.

### How to read the table (this is the key step)
| Column | Good result |
|---|---|
| **Strat Sharpe vs B&H Sharpe** (primary) | Strat **higher** on most assets |
| **Strat MaxDD vs B&H MaxDD** | Strat **shallower** (less negative) |
| Strat ROI vs B&H ROI | May trail B&H — **expected**, you're de-risked on average |
| Avg Expo | **< 1.0** confirms risk was cut in turbulent periods |

The summary line **"Sharpe improved vs B&H on X/N assets"** is the headline.

In [ ]:
# ==========================================
# STEP 3: VOLATILITY-MANAGED BACKTEST vs BUY-AND-HOLD
# ==========================================
print("--- VOL-MANAGED BACKTEST ---")
!python src/vol_targeting.py --config config/config_phase1.json

---
## STEP 3b: Full-History Evaluation (all regimes)

The STEP 3 result above used only the recent ~20% test window — which on the first run was a strong equity **bull**, vol targeting's *worst* regime (de-risking just caps upside you didn't need to protect). This re-runs the backtest over the **entire train+test history** (all regimes), fully causal: EWMA forecast shifted one bar + an **expanding-median** target, so it only ever sees the past (flat during a 250-bar warmup).

Using *all* the data is the opposite of cherry-picking a favourable window. **PBO remains the final arbiter** of robustness — this is just a fair read before we gate.

Read it the same way: **Sharpe & MaxDD vs B&H**, and the mean Sharpe line.

In [ ]:
# ==========================================
# STEP 3b: FULL-HISTORY EVALUATION (all regimes)
# ==========================================
# Re-runs the backtest over the ENTIRE history.
!python src/vol_targeting.py --config config/config_phase1.json --full

---
## STEP 3c: Leverage Sweep — Sharpe gain vs drawdown cost

The no-leverage result (3b) cut drawdown but did **not** improve Sharpe — because the Sharpe gain in vol targeting comes from **levering UP in calm periods**, which `vol_cap = 1.0` disabled. This sweeps `vol_cap` from 1.0 → 2.0 over the full history, with a **financing cost** on the borrowed portion (5%/yr) so leverage isn't free.

**Read the curve:**
- **`vs B&H` column** — does mean Strat Sharpe rise **above** Buy-and-Hold as the cap increases?
- **`mean Strat MaxDD%`** — how much **deeper** does drawdown get as you lever up? That is the cost of relaxing capital protection.

The decision is the *tradeoff*: how much Sharpe does leverage buy, and is the added drawdown acceptable? **PBO still gates** any config before it's real.

In [ ]:
# ==========================================
# STEP 3c: LEVERAGE SWEEP (Sharpe gain vs drawdown cost)
# ==========================================
# Relaxes the no-leverage cap. Sweeps vol_cap 1.0 -> 2.0 over the full history,
# charging a financing cost on the borrowed portion (--borrow_rate, default 5%/yr).
!python src/vol_targeting.py --config config/config_phase1.json --sweep

---
## Read & Next Steps

- **Sharpe beats B&H on most assets** → the premise holds. Next: build the
  **vol-targeting PBO gate** (vary the EWMA halflife / target as the trial grid)
  to confirm it's robust, not a parameter fluke — target **PBO ≤ 0.20**. Then,
  optionally, an ML volatility forecast — *only if it still clears PBO*.
- **Sharpe doesn't beat B&H** → the premise doesn't hold on this window; we
  reconsider the universe / horizon / forecast.

Experiment with the risk knob and forecast horizon:
```python
!python src/vol_targeting.py --config config/config_phase1.json --halflife 40 --vol_cap 1.0
```

**Remember:** judge on **Sharpe & drawdown vs Buy-and-Hold**, not ROI. The outcome of
this run is written up in the README's Results section.